# 02 · Extract fields and classify the drawings with `ai_extract` and `ai_classify`

Second step of the document-intelligence chain (parse, then extract and classify, then check compliance). It reads the parsed drawings from `bluerridge_silver_engineering.drawings_parsed`, sends each drawing's `parsed_markdown` to the built-in `ai_extract` and `ai_classify` AI functions, and writes one row per drawing to `bluerridge_silver_engineering.drawings_extracted`:

| column | type | what it holds |
|---|---|---|
| `drawing_id` | STRING | the file name without `.pdf`; joins to `drawings_parsed` |
| `title_block_number` | STRING | the drawing number printed in the title block, read by `ai_extract` (it should equal `drawing_id`) |
| `discipline` | STRING | `mechanical`, `electrical`, `structural`, `piping` or `instr`: the title block's DISCIPLINE word, normalized |
| `revision` | STRING | the current revision, the letter in the title block's REV box |
| `drawing_type` | STRING | `ISO`, `PID`, `GA`, `SPEC` or `OTHER`, from `ai_classify` |
| `spec_citations` | `ARRAY<STRING>` | the codes and standards the drawing cites, for example `ASME B31.3 §322`; an empty array when it cites none |
| `extract_status` | STRING | `ok`, or `error:<reason>` |
| `classify_status` | STRING | `ok`, or `error:<reason>` |
| `extracted_at` | TIMESTAMP | when the AI step started (one value for the whole run) |

How it runs:

- Each function is called once per drawing, written as a Spark expression, so the calls spread over the compute instead of running in a Python loop. Neither function takes a model argument (checked below), so the service's default models do the work.
- The AI step runs once, in the cell that writes `drawings_extracted_raw`: the two answers for every drawing, exactly as the functions returned them. Everything after it (the cleaning, `drawings_extracted`, the checks and the sample) reads that table, so no drawing is sent to a model twice and changing a cleaning rule costs nothing.
- Running the notebook again sends all 1,200 drawings through both functions again, which is 2,400 calls and a few million input tokens, and replaces both tables. The job that runs it must switch off serverless auto-retry (`disable_auto_optimization`, see `docs/BUILD.md`), or a failure would repeat every call.
- A positive `max_rows` widget sends only that many drawings and writes `drawings_extracted_raw_dryrun` and `drawings_extracted_dryrun` instead, so a trial run cannot replace the real tables.
- The answers come from models, so a re-run can differ a little from the run recorded here.

In [1]:
import contextlib
import io
import json
import re
import time
from pathlib import Path

from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType, StructField, StructType

# Widgets let a job override these; the defaults match the variables in databricks.yml.
dbutils.widgets.text("catalog", "mfg_mc_se_sa", "Catalog")
dbutils.widgets.text("schema_prefix", "bluerridge", "Schema prefix")
# 0 sends every drawing to the models. A positive number sends only that many (a fixed pseudo-random subset): a trial run
# that writes <table>_dryrun, so it can never replace the real 1,200-row tables.
dbutils.widgets.text("max_rows", "0", "Max drawings to send to the models (0 = all)")
catalog = dbutils.widgets.get("catalog")
prefix = dbutils.widgets.get("schema_prefix")
MAX_ROWS = int(dbutils.widgets.get("max_rows") or 0)
SUFFIX = "_dryrun" if MAX_ROWS else ""

SILVER = f"{catalog}.{prefix}_silver_engineering"
PARSED = f"{SILVER}.drawings_parsed"
RAW = f"{SILVER}.drawings_extracted_raw{SUFFIX}"
TARGET = f"{SILVER}.drawings_extracted{SUFFIX}"

LABELS = ["ISO", "PID", "GA", "SPEC", "OTHER"]  # the drawing_type values
DISCIPLINES = ["mechanical", "electrical", "structural", "piping", "instr"]  # the discipline values the project data uses
PARTITIONS = 32  # tasks that make the AI calls side by side
SAMPLE_ID = "BRL-INS-PID-0052"  # the P&ID with three planted flaws that the parse notebook wrote out in full
OUT = Path("outputs")  # next to this notebook


def sql_literal(text):
    # `text` as a SQL string literal (backslashes and single quotes escaped).
    return "'" + text.replace("\\", "\\\\").replace("'", "\\'") + "'"


print(f"source: {PARSED}")
print(f"the models' answers: {RAW}")
print(f"target: {TARGET}")
print(f"drawings to send: {MAX_ROWS or 'all'}" + (" (trial run: the real tables are not touched)" if MAX_ROWS else ""))

source: mfg_mc_se_sa.bluerridge_silver_engineering.drawings_parsed
the models' answers: mfg_mc_se_sa.bluerridge_silver_engineering.drawings_extracted_raw
target: mfg_mc_se_sa.bluerridge_silver_engineering.drawings_extracted
drawings to send: all


## Which call signature works?

The plan writes the calls as `ai_extract(parsed_markdown, schema => '{title_block_number: string, ...}')` and `ai_classify(parsed_markdown, labels => array(...))`, and prefers a cheaper model if a function accepts one. Each of those forms is tried on one drawing (`BRL-INS-PID-0052`), next to the forms the function reference describes. A bad argument is refused when the AI function is compiled, before a model sees anything; each form that is accepted makes one real call on that single drawing.

In [2]:
SCHEMA_JSON = json.dumps({
    "title_block_number": {"type": "string"},
    "revision": {"type": "string"},
    "discipline": {"type": "string"},
    "spec_citations": {"type": "array", "items": {"type": "string"}},
})
LABELS_JSON = json.dumps(LABELS)
HAIKU = "databricks-claude-haiku-4-5"


def tried(call):
    # (True, the start of the answer) if `SELECT <call>` runs on the sample drawing, else (False, the error's first line).
    try:
        row = spark.sql(f"SELECT CAST({call} AS STRING) AS answer FROM {PARSED} WHERE drawing_id = '{SAMPLE_ID}'").first()
        return True, (row.answer or "")[:150]
    except Exception as e:
        return False, str(e).strip().splitlines()[0][:230]


FORMS = (
    ("ai_extract", "the plan's form, a schema written as {key: type}",
     "ai_extract(parsed_markdown, schema => '{title_block_number: string, revision: string, discipline: string, spec_citations: array<string>}')"),
    ("ai_extract", "a schema as column definitions",
     "ai_extract(parsed_markdown, 'schema=title_block_number STRING, revision STRING, discipline STRING, spec_citations ARRAY<STRING>')"),
    ("ai_extract", "a JSON schema, no options",
     f"ai_extract(parsed_markdown, {sql_literal(SCHEMA_JSON)})"),
    ("ai_extract", "a JSON schema, options map with version 2.0",
     f"ai_extract(parsed_markdown, {sql_literal(SCHEMA_JSON)}, map('version', '2.0'))"),
    ("ai_extract", f"a JSON schema, model => '{HAIKU}'",
     f"ai_extract(parsed_markdown, {sql_literal(SCHEMA_JSON)}, model => '{HAIKU}')"),
    ("ai_extract", "a JSON schema, the model name as the third argument",
     f"ai_extract(parsed_markdown, {sql_literal(SCHEMA_JSON)}, '{HAIKU}')"),
    ("ai_classify", "the plan's form, labels => array(...)",
     "ai_classify(parsed_markdown, labels => array('ISO', 'PID', 'GA', 'SPEC', 'OTHER'))"),
    ("ai_classify", "array(...) as the second argument",
     "ai_classify(parsed_markdown, array('ISO', 'PID', 'GA', 'SPEC', 'OTHER'))"),
    ("ai_classify", "JSON labels, options map with version 2.0",
     f"ai_classify(parsed_markdown, {sql_literal(LABELS_JSON)}, map('version', '2.0'))"),
    ("ai_classify", f"JSON labels, model => '{HAIKU}'",
     f"ai_classify(parsed_markdown, {sql_literal(LABELS_JSON)}, model => '{HAIKU}')"),
    ("ai_classify", "JSON labels, the model name as the third argument",
     f"ai_classify(parsed_markdown, {sql_literal(LABELS_JSON)}, '{HAIKU}')"),
)
for function, what, call in FORMS:
    ok, text = tried(call)
    print(f"{'accepted' if ok else 'rejected'}  {function}: {what}")
    print(f"          {text}")

rejected  ai_extract: the plan's form, a schema written as {key: type}
          [AI_FUNCTION_COMPILATION_ERROR] An error occurred while compiling the AI function: INVALID_ARGUMENT: Could not parse the provided schema: Expecting property name enclosed in double quotes: line 1 column 2 (char 1). SQLSTATE: 42883
rejected  ai_extract: a schema as column definitions
          [AI_FUNCTION_COMPILATION_ERROR] An error occurred while compiling the AI function: INVALID_ARGUMENT: Could not parse the provided schema: Expecting value: line 1 column 1 (char 0). SQLSTATE: 42883; line 1 pos 12
accepted  ai_extract: a JSON schema, no options
          {"error_message":null,"metadata":{"version":"2.1"},"response":{"discipline":{"value":"INSTR"},"revision":{"value":"B"},"spec_citations":[{"value":"ASM
accepted  ai_extract: a JSON schema, options map with version 2.0
          {"error_message":null,"metadata":{"version":"2.0"},"response":{"discipline":"INSTR","revision":"B","spec_citations":["ASME B31.3

Both functions take their definition as a JSON string plus an optional options map: `ai_extract` a schema (`{"field": {"type": "string"}, ...}`, where a field can also carry a `description`), `ai_classify` its labels (`["ISO", ...]`, or `{"ISO": "what it is", ...}`). Neither takes a model, so, as with `ai_parse_document`, the service default does the work and there is no cheaper model to choose. Three details decide the forms used below:

- `ai_extract` without options is version 2.1, which wraps every value as `{"value": ...}` to carry citation and confidence data this table does not use. Version 2.0 returns the plain values, so the notebook pins it.
- `ai_classify` also accepts `array('ISO', ...)`, but that older form returns only the bare label. The JSON form returns the label inside a VARIANT next to an `error_message`, which becomes `classify_status`, and takes a description for each label.
- A schema written as column definitions, or as JSON with unquoted keys, is not valid JSON and is refused.

## Pick the documents

Rows of `drawings_parsed` with `parse_status = 'ok'`; a drawing that did not parse has no text to send.

In [3]:
parsed_all = spark.table(PARSED)
n_parsed = parsed_all.count()
docs = parsed_all.where("parse_status = 'ok'").select("drawing_id", "parsed_markdown")
n_ok = docs.count()
print(f"drawings_parsed rows: {n_parsed}; with parse_status ok: {n_ok}; left out because the parse failed: {n_parsed - n_ok}")
assert n_ok > 0

n_docs = n_ok
if MAX_ROWS:
    # A fixed pseudo-random subset (the same on every run) that always includes the sample drawing.
    docs = docs.orderBy(F.col("drawing_id") != SAMPLE_ID, F.xxhash64("drawing_id")).limit(MAX_ROWS)
    n_docs = min(MAX_ROWS, n_ok)
print(f"drawings to send to the models: {n_docs}")

drawings_parsed rows: 1200; with parse_status ok: 1200; left out because the parse failed: 0
drawings to send to the models: 1200


## The `ai_extract` call

The schema names the four fields and says where to find each one, because the layout of the text is what goes wrong otherwise: the drawing number sits in a different table cell on different drawing types, there are two revision tables (the revision history and the title block), and the cited standards are spread over a reference list, the notes and the table headings. `discipline` is read as a free string, exactly as printed, and mapped to the five project values afterwards in plain code that can be checked (the cleaning cell below); a typed `enum` would hide that mapping inside the model.

In [4]:
EXTRACT_SCHEMA = {
    "title_block_number": {
        "type": "string",
        "description": "Drawing number in the title block DRAWING NUMBER box, for example BRL-PIP-ISO-0010. Not the sheet, a line number or an equipment tag.",
    },
    "revision": {
        "type": "string",
        "description": "Current revision: the letter in the title block REV box, not a row of the revision history table.",
    },
    "discipline": {
        "type": "string",
        "description": "The word in the title block DISCIPLINE box, as printed.",
    },
    "spec_citations": {
        "type": "array",
        "items": {"type": "string"},
        "description": (
            "Codes and standards the drawing cites, one entry per distinct reference, written as printed but without the title of the standard: "
            "the designation, plus the clause when the text gives one. Examples of entries: ASME B16.5, ASME B31.3 PARA. 322, "
            "OSHA 29 CFR 1910.147, IBC 1005.1. A reference list line such as ASME B31.3 PROCESS PIPING or OSHA 29 CFR 1910.147 LOCKOUT/TAGOUT "
            "gives the entry ASME B31.3 or OSHA 29 CFR 1910.147. Take them from the REFERENCE CODES AND STANDARDS list, the general notes "
            "and the table headings. Leave out any material group, pressure or temperature."
        ),
    },
}
EXTRACT_INSTRUCTIONS = (
    "Each text is one drawing of the Blue Ridge LNG terminal, read from a PDF. The title block is the table near the end of the text "
    "with PROJECT, TITLE, DRAWING NUMBER, REV, SCALE, SHEET, DATE and DISCIPLINE. Take the drawing number, the revision and the "
    "discipline from the title block. A REV / DATE / DESCRIPTION table lists the earlier revisions as well; the revision wanted is the "
    "one in the title block REV box."
)
EXTRACT_CALL = (
    f"ai_extract(parsed_markdown, {sql_literal(json.dumps(EXTRACT_SCHEMA))}, "
    f"map('version', '2.0', 'instructions', {sql_literal(EXTRACT_INSTRUCTIONS)}))"
)
print("fields:", ", ".join(EXTRACT_SCHEMA))
print("instructions:", EXTRACT_INSTRUCTIONS)

fields: title_block_number, revision, discipline, spec_citations
instructions: Each text is one drawing of the Blue Ridge LNG terminal, read from a PDF. The title block is the table near the end of the text with PROJECT, TITLE, DRAWING NUMBER, REV, SCALE, SHEET, DATE and DISCIPLINE. Take the drawing number, the revision and the discipline from the title block. A REV / DATE / DESCRIPTION table lists the earlier revisions as well; the revision wanted is the one in the title block REV box.


## The `ai_classify` call

The five labels each come with a one-line description, which is how the function learns what a label means: `OTHER`, in particular, is only a meaningful class once it is described. The answer is meant to be one of the five labels, and the cleaning step below marks anything else as an error.

In [5]:
LABEL_DESCRIPTIONS = {
    "ISO": "Piping isometric of one line: line data table, bill of materials, weld table and a dimensioned single-line sketch",
    "PID": "Piping and instrumentation diagram: process schematic with equipment tags, instrument tags, line numbers and flange classes",
    "GA": "General arrangement: plan or elevation of a building or area with grid lines, equipment footprints, exits and an egress schedule",
    "SPEC": "Specification sheet: tables of piping classes, weld requirements or relief device schedules, with notes and no drawing of a line or area",
    "OTHER": "Any other sheet, such as a standard symbols and legend sheet or a typical detail",
}
assert list(LABEL_DESCRIPTIONS) == LABELS
CLASSIFY_INSTRUCTIONS = (
    "Each text is one engineering drawing for an LNG terminal, read from a PDF. "
    "The TITLE in the title block says what kind of drawing it is."
)
CLASSIFY_CALL = (
    f"ai_classify(parsed_markdown, {sql_literal(json.dumps(LABEL_DESCRIPTIONS))}, "
    f"map('version', '2.0', 'instructions', {sql_literal(CLASSIFY_INSTRUCTIONS)}))"
)
for label, meaning in LABEL_DESCRIPTIONS.items():
    print(f"{label:5} {meaning}")

ISO   Piping isometric of one line: line data table, bill of materials, weld table and a dimensioned single-line sketch
PID   Piping and instrumentation diagram: process schematic with equipment tags, instrument tags, line numbers and flange classes
GA    General arrangement: plan or elevation of a building or area with grid lines, equipment footprints, exits and an egress schedule
SPEC  Specification sheet: tables of piping classes, weld requirements or relief device schedules, with notes and no drawing of a line or area
OTHER Any other sheet, such as a standard symbols and legend sheet or a typical detail


## Both calls in one plan

Nothing runs yet: a DataFrame is a plan. `repartition` spreads the drawings over several tasks, because a table this small would otherwise be read as one or two partitions and the calls would run in a line. The answers are cast to JSON strings inside the SQL expression, so the table holds plain strings and the Python side never handles a VARIANT. The plan is checked before anything is written: it must hold exactly two `AI Query` steps, one `ai_extract` and one `ai_classify`, so that no drawing goes to a function twice.

In [6]:
ai_df = (
    docs.repartition(PARTITIONS)
    .withColumn("extract_json", F.expr(f"CAST({EXTRACT_CALL} AS STRING)"))
    .withColumn("classify_json", F.expr(f"CAST({CLASSIFY_CALL} AS STRING)"))
    .select("drawing_id", "extract_json", "classify_json", F.current_timestamp().alias("extracted_at"))
)
print("columns:", ai_df.columns)

# explain() only plans the query; it does not run it. Only the physical plan is read: the "== Photon Explanation ==" after it
# quotes the AI step again and would be counted twice.
plan = io.StringIO()
with contextlib.redirect_stdout(plan):
    ai_df.explain(mode="formatted")
physical_plan = re.split(r"^== (?!Physical Plan)", plan.getvalue(), flags=re.M)[0]
ai_steps = re.findall(r"^\(\d+\) AI Query.*$", physical_plan, flags=re.M)
ai_functions = re.findall(r", (ai_\w+), VariantType\)", physical_plan)
print(f"AI Query steps in the physical plan: {len(ai_steps)} ({', '.join(ai_functions)})")
assert len(ai_steps) == 2 and sorted(ai_functions) == ["ai_classify", "ai_extract"], "each AI function should run exactly once per drawing"

columns: ['drawing_id', 'extract_json', 'classify_json', 'extracted_at']
AI Query steps in the physical plan: 2 (ai_extract, ai_classify)


## Send the drawings to the models and write their answers

The only cell that calls the AI functions. `overwrite` makes a re-run replace the table instead of appending a second copy of every drawing.

In [7]:
t0 = time.time()
ai_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(RAW)
print(f"sent {n_docs} drawings to ai_extract and ai_classify and wrote {RAW} in {time.time() - t0:.0f} s")

sent 1200 drawings to ai_extract and ai_classify and wrote mfg_mc_se_sa.bluerridge_silver_engineering.drawings_extracted_raw in 355 s


## Read the answers back

From here on the notebook reads this table, never the DataFrame above, so nothing is sent to a model again. Three drawings, with the answers exactly as the functions returned them (`response` holds the result, `error_message` is null when the call worked):

In [8]:
print(f"{spark.table(RAW).count()} rows in {RAW}")
display(spark.table(RAW).orderBy(F.rand(seed=42)).limit(3))

1200 rows in mfg_mc_se_sa.bluerridge_silver_engineering.drawings_extracted_raw


drawing_id,extract_json,classify_json,extracted_at
BRL-INS-PID-0566,"{""error_message"":null,""metadata"":{""version"":""2.0""},""response"":{""discipline"":""INSTR"",""revision"":""C"",""spec_citations"":[""ASME B31.3 PARA. 322"",""ASME B16.5"",""OSHA 29 CFR 1910.147"",""ISA 5.1""],""title_block_number"":""BRL-INS-PID-0566""}}","{""error_message"":null,""metadata"":{""version"":""2.0""},""response"":[""PID""]}",2026-11-07T05:47:11.254Z
BRL-MEC-SPC-0909,"{""error_message"":null,""metadata"":{""version"":""2.0""},""response"":{""discipline"":""MECHANICAL"",""revision"":""E"",""spec_citations"":[""ASME B31.3 PARA. 341"",""ASME B31.3 PARA. 322"",""ASME B16.5"",""API 526""],""title_block_number"":""BRL-MEC-SPC-0909""}}","{""error_message"":null,""metadata"":{""version"":""2.0""},""response"":[""SPEC""]}",2026-11-07T05:47:11.254Z
BRL-PIP-SPC-0966,"{""error_message"":null,""metadata"":{""version"":""2.0""},""response"":{""discipline"":""PIPING"",""revision"":""C"",""spec_citations"":[""ASME B31.3"",""ASME B31.3 PARA. 341"",""ASME B31.3 PARA. 322"",""ASME B16.5"",""API 526""],""title_block_number"":""BRL-PIP-SPC-0966""}}","{""error_message"":null,""metadata"":{""version"":""2.0""},""response"":[""SPEC""]}",2026-11-07T05:47:11.254Z


## Clean the answers into columns

Plain functions turn one drawing's two JSON answers into column values. They never raise, because one odd answer must not fail a run that has already paid for all the others.

- `read_extract` and `read_classify` take the answers apart. A row gets `error:<reason>` when the function reports an `error_message`, returns nothing, or (`ai_extract`) finds none of the fields, or (`ai_classify`) does not return exactly one of the five labels.
- `normalize_discipline` maps the title block's word to the five project values: lower-cased, then matched on its first letters (`INSTRUMENTATION` and `INSTR` are both `instr`) or on the three-letter code used in drawing numbers (`INS`, `ELE`, ...). A word that matches none of them becomes NULL, never a guess.
- `normalize_revision` drops a leading `REV` and upper-cases the letter.
- `clean_citations` collapses white space, writes a clause as `§` (`ASME B31.3 PARA. 322` becomes `ASME B31.3 §322`, the form the compliance ground truth and the judge use) and drops duplicates.

They are tried on hand-made answers first, including cases the real data may not produce.

In [9]:
def one_line(value):
    # A value as one trimmed line of text, or None when it is missing, blank or not text.
    if not isinstance(value, (str, int, float)) or isinstance(value, bool):
        return None
    return " ".join(str(value).split()) or None


def read_extract(extract_json):
    # (title_block_number, discipline_raw, revision_raw, citations_raw, extract_status) from one ai_extract answer.
    try:
        if not extract_json:
            return None, None, None, [], "error:no result"
        answer = json.loads(extract_json)
        if answer.get("error_message"):
            return None, None, None, [], f"error:{answer['error_message']}"[:300]
        fields = answer.get("response") or {}
        number, discipline, revision = (one_line(fields.get(key)) for key in ("title_block_number", "discipline", "revision"))
        cited = fields.get("spec_citations")
        citations = [c for c in cited if isinstance(c, str)] if isinstance(cited, list) else []
        if not (number or discipline or revision or citations):
            return None, None, None, [], "error:no fields extracted"
        return number, discipline, revision, citations, "ok"
    except Exception as e:
        return None, None, None, [], f"error:unreadable answer ({type(e).__name__})"


def read_classify(classify_json):
    # (drawing_type, classify_status) from one ai_classify answer.
    try:
        if not classify_json:
            return None, "error:no result"
        answer = json.loads(classify_json)
        if answer.get("error_message"):
            return None, f"error:{answer['error_message']}"[:300]
        found = answer.get("response")
        if not isinstance(found, list) or len(found) != 1:
            return None, f"error:expected one label, got {found}"[:300]
        label = str(found[0]).strip()
        if label not in LABELS:
            return None, f"error:unexpected label {label}"[:300]
        return label, "ok"
    except Exception as e:
        return None, f"error:unreadable answer ({type(e).__name__})"


DISCIPLINE_STARTS = {"mech": "mechanical", "elec": "electrical", "struct": "structural", "pip": "piping", "instr": "instr"}
DISCIPLINE_CODES = {"mec": "mechanical", "ele": "electrical", "str": "structural", "ins": "instr"}  # the codes in drawing numbers (PIP is a start)
CLAUSE_MARK = re.compile(r"\s+(?:PARAGRAPH|PARA\.?|SECTION|SEC\.?|§)\s*(?=\d)", re.I)


def normalize_discipline(word):
    # The title block's DISCIPLINE word as one of the five project values, or None when it is not one of them.
    if not isinstance(word, str):
        return None
    first = (re.findall(r"[a-z]+", word.lower()) or [""])[0]
    if first in DISCIPLINES:
        return first
    if first in DISCIPLINE_CODES:
        return DISCIPLINE_CODES[first]
    return next((value for start, value in DISCIPLINE_STARTS.items() if first.startswith(start)), None)


def normalize_revision(text):
    # "B", "rev b" and "Rev. B" all become "B"; None when blank.
    if not isinstance(text, str):
        return None
    return re.sub(r"^(?:REVISION|REV)\b\.?\s*", "", text.strip(), flags=re.I).strip().upper() or None


def clean_citation(text):
    # One citation on one line with its clause written as § (ASME B31.3 PARA. 322 -> ASME B31.3 §322); None when blank.
    if not isinstance(text, str):
        return None
    return CLAUSE_MARK.sub(" §", " ".join(text.split())).strip(" ,;") or None


def clean_citations(items):
    # The distinct citations, in the order given, each cleaned.
    seen, cleaned = set(), []
    for item in items or []:
        text = clean_citation(item)
        if text and text.casefold() not in seen:
            seen.add(text.casefold())
            cleaned.append(text)
    return cleaned


# Hand-made answers: a good one, a failed call, nothing returned, no fields, and something that is not JSON.
good = json.dumps({"error_message": None, "response": {"title_block_number": "BRL-PIP-ISO-0010", "revision": "C", "discipline": "PIPING",
                                                          "spec_citations": ["ASME B31.3", "ASME B31.3 PARA. 322"]}})
failed = json.dumps({"error_message": "rate limited", "response": None})
nothing = json.dumps({"error_message": None, "response": {"title_block_number": None, "revision": "", "discipline": None, "spec_citations": []}})
for name, value in (("good", good), ("failed", failed), ("nothing", nothing), ("None", None), ("not JSON", "<html>")):
    print(f"{name:9} -> {read_extract(value)}")
assert read_extract(good) == ("BRL-PIP-ISO-0010", "PIPING", "C", ["ASME B31.3", "ASME B31.3 PARA. 322"], "ok")
assert read_extract(failed)[4] == "error:rate limited"
assert read_extract(nothing)[4] == "error:no fields extracted"
assert read_extract(None)[4] == "error:no result" and read_extract("<html>")[4].startswith("error:unreadable answer")

for name, value in (("one label", {"response": ["PID"], "error_message": None}), ("two labels", {"response": ["PID", "GA"], "error_message": None}),
                    ("new label", {"response": ["DWG"], "error_message": None}), ("failed", {"response": None, "error_message": "timeout"})):
    print(f"{name:9} -> {read_classify(json.dumps(value))}")
assert read_classify(json.dumps({"response": ["PID"], "error_message": None})) == ("PID", "ok")
assert read_classify(json.dumps({"response": ["PID", "GA"], "error_message": None}))[0] is None
assert read_classify(json.dumps({"response": ["DWG"], "error_message": None})) == (None, "error:unexpected label DWG")
assert read_classify(json.dumps({"response": None, "error_message": "timeout"})) == (None, "error:timeout")

DISCIPLINE_CASES = {"PIPING": "piping", "instr": "instr", "Instrumentation": "instr", "INSTR.": "instr", "Instrument & Control": "instr", " mech ": "mechanical",
                    "Electrical": "electrical", "STRUCTURAL": "structural", "ELE": "electrical", "Process": None, "Insulation": None, "": None, None: None}
for word, expected in DISCIPLINE_CASES.items():
    assert normalize_discipline(word) == expected, (word, normalize_discipline(word), expected)
print(f"normalize_discipline: {len(DISCIPLINE_CASES)} hand-made words ok, for example 'Instrumentation' -> {normalize_discipline('Instrumentation')!r}, 'Process' -> {normalize_discipline('Process')!r}")
assert [normalize_revision(t) for t in ("B", "rev b", "Rev. C", "REVISION D", "", None)] == ["B", "B", "C", "D", None, None]

cited = ["ASME B31.3 PARA. 322", "ASME  B31.3", "asme b31.3", "", None, "ASME B31.3 §322", "OSHA 29 CFR 1910.147", "IBC SECTION 1005.1", "ASME B16.5, SS GROUP 2.1"]
print("clean_citations ->", clean_citations(cited))
assert clean_citations(cited) == ["ASME B31.3 §322", "ASME B31.3", "OSHA 29 CFR 1910.147", "IBC §1005.1", "ASME B16.5, SS GROUP 2.1"]
assert clean_citations(None) == [] and clean_citations([]) == []

good      -> ('BRL-PIP-ISO-0010', 'PIPING', 'C', ['ASME B31.3', 'ASME B31.3 PARA. 322'], 'ok')
failed    -> (None, None, None, [], 'error:rate limited')
nothing   -> (None, None, None, [], 'error:no fields extracted')
None      -> (None, None, None, [], 'error:no result')
not JSON  -> (None, None, None, [], 'error:unreadable answer (JSONDecodeError)')
one label -> ('PID', 'ok')
two labels -> (None, "error:expected one label, got ['PID', 'GA']")
new label -> (None, 'error:unexpected label DWG')
failed    -> (None, 'error:timeout')
normalize_discipline: 13 hand-made words ok, for example 'Instrumentation' -> 'instr', 'Process' -> None
clean_citations -> ['ASME B31.3 §322', 'ASME B31.3', 'OSHA 29 CFR 1910.147', 'IBC §1005.1', 'ASME B16.5, SS GROUP 2.1']


## Build `drawings_extracted`

The answers table, cleaned into the nine columns. Nothing here calls an AI function. The first audit shows what the normalization did to the real `discipline` words; the second lists the citations that cleaning changed.

In [10]:
extract_type = StructType([
    StructField("title_block_number", StringType()),
    StructField("discipline_raw", StringType()),
    StructField("revision_raw", StringType()),
    StructField("citations_raw", ArrayType(StringType())),
    StructField("extract_status", StringType()),
])
classify_type = StructType([StructField("drawing_type", StringType()), StructField("classify_status", StringType())])

read_extract_udf = F.udf(read_extract, extract_type)
read_classify_udf = F.udf(read_classify, classify_type)
discipline_udf = F.udf(normalize_discipline, "string")
revision_udf = F.udf(normalize_revision, "string")
citation_udf = F.udf(clean_citation, "string")
citations_udf = F.udf(clean_citations, "array<string>")

answers = (
    spark.table(RAW)
    .withColumn("x", read_extract_udf("extract_json"))
    .withColumn("c", read_classify_udf("classify_json"))
)
extracted = answers.select(
    "drawing_id",
    F.col("x.title_block_number").alias("title_block_number"),
    discipline_udf("x.discipline_raw").alias("discipline"),
    revision_udf("x.revision_raw").alias("revision"),
    F.col("c.drawing_type").alias("drawing_type"),
    citations_udf("x.citations_raw").alias("spec_citations"),
    F.col("x.extract_status").alias("extract_status"),
    F.col("c.classify_status").alias("classify_status"),
    "extracted_at",
)
print("columns:", extracted.columns)

columns: ['drawing_id', 'title_block_number', 'discipline', 'revision', 'drawing_type', 'spec_citations', 'extract_status', 'classify_status', 'extracted_at']


In [11]:
# What the normalization did to the real discipline words: the word the model returned, and the value it became.
words = answers.select(F.col("x.discipline_raw").alias("title_block_word"), discipline_udf("x.discipline_raw").alias("discipline"))
display(words.groupBy("title_block_word", "discipline").count().orderBy("discipline", "title_block_word"))
w = words.agg(
    F.count("*").alias("rows"),
    F.sum(F.when(F.col("title_block_word").isNull(), 1).otherwise(0)).alias("no_word"),
    F.sum(F.when(F.col("discipline").isNull() & F.col("title_block_word").isNotNull(), 1).otherwise(0)).alias("not_recognised"),
    F.sum(F.when(F.lower(F.trim("title_block_word")) == F.col("discipline"), 1).otherwise(0)).alias("lower_case_only"),
    F.sum(F.when(F.col("discipline").isNotNull() & (F.lower(F.trim("title_block_word")) != F.col("discipline")), 1).otherwise(0)).alias("other_spelling"),
).first()
print(f"{w.rows} rows: {w.lower_case_only} needed only lower-casing, {w.other_spelling} were mapped from another spelling, "
      f"{w.not_recognised} word(s) not recognised (NULL), {w.no_word} with no word")

# The citations cleaning changed: as the model wrote them, and as stored.
changed = (
    answers.select(F.explode("x.citations_raw").alias("as_extracted"))
    .withColumn("stored", citation_udf("as_extracted"))
    .where("stored IS NULL OR stored <> as_extracted")
    .groupBy("as_extracted", "stored").count()
    .orderBy(F.desc("count"), "as_extracted")
    .limit(10)
)
display(changed)

title_block_word,discipline,count
DISCIPINE,,1
INST,,1
ELECTRICAL,electrical,77
INSTR,instr,119
MECHANICAL,mechanical,134
PIPING,piping,755
STRUCTURAL,structural,113


1200 rows: 1198 needed only lower-casing, 0 were mapped from another spelling, 2 word(s) not recognised (NULL), 0 with no word


as_extracted,stored,count
ASME B31.3 PARA. 341,ASME B31.3 §341,516
ASME B31.3 PARA. 322,ASME B31.3 §322,448


Write the table and document it in Unity Catalog (a table comment, column comments, `quality = silver` like the other silver tables), then show the schema it ended up with:

In [12]:
t0 = time.time()
extracted.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET)
print(f"wrote {TARGET} in {time.time() - t0:.0f} s")

spark.sql(
    f"COMMENT ON TABLE {TARGET} IS 'One row per drawing: title block fields and citations read by ai_extract, drawing type from ai_classify "
    "(service default models). Built from drawings_extracted_raw by 02_extract_classify.'"
)
spark.sql(
    f"COMMENT ON TABLE {RAW} IS 'The ai_extract and ai_classify answers exactly as the functions returned them (JSON), one row per drawing. "
    "drawings_extracted is built from this table without further AI calls.'"
)
COLUMN_COMMENTS = {
    "drawing_id": "File name without .pdf; key into drawings_parsed",
    "title_block_number": "Drawing number printed in the title block, read by ai_extract; should equal drawing_id",
    "discipline": "mechanical, electrical, structural, piping or instr: the title block word, normalized; NULL when not recognised",
    "revision": "Current revision: the letter in the title block REV box",
    "drawing_type": "ISO, PID, GA, SPEC or OTHER, from ai_classify",
    "spec_citations": "Codes and standards the drawing cites, for example ASME B31.3 §322; empty array when none",
    "extract_status": "ok, or error:<reason>",
    "classify_status": "ok, or error:<reason>",
    "extracted_at": "When the ai_extract and ai_classify step started (one value for the whole run)",
}
for column, comment in COLUMN_COMMENTS.items():
    spark.sql(f"ALTER TABLE {TARGET} ALTER COLUMN {column} COMMENT {sql_literal(comment)}")
spark.sql(f"ALTER TABLE {TARGET} SET TBLPROPERTIES ('quality' = 'silver')")

display(spark.sql(f"""
    SELECT ordinal_position AS position, column_name, full_data_type AS data_type, comment
    FROM {catalog}.information_schema.columns
    WHERE table_schema = '{prefix}_silver_engineering' AND table_name = '{TARGET.split('.')[-1]}'
    ORDER BY ordinal_position
"""))

wrote mfg_mc_se_sa.bluerridge_silver_engineering.drawings_extracted in 3 s


position,column_name,data_type,comment
0,drawing_id,string,File name without .pdf; key into drawings_parsed
1,title_block_number,string,"Drawing number printed in the title block, read by ai_extract; should equal drawing_id"
2,discipline,string,"mechanical, electrical, structural, piping or instr: the title block word, normalized; NULL when not recognised"
3,revision,string,Current revision: the letter in the title block REV box
4,drawing_type,string,"ISO, PID, GA, SPEC or OTHER, from ai_classify"
5,spec_citations,array<string>,"Codes and standards the drawing cites, for example ASME B31.3 §322; empty array when none"
6,extract_status,string,"ok, or error:<reason>"
7,classify_status,string,"ok, or error:<reason>"
8,extracted_at,timestamp,When the ai_extract and ai_classify step started (one value for the whole run)


## Verify

The table's shape and the rules from the data model, as assertions: one row per drawing and none for a drawing that is not in `drawings_parsed`, `discipline` one of the five values or NULL (and set for at least 95% of the drawings), `drawing_type` one of the five labels whenever the classification worked, `spec_citations` an array (empty is fine), and no more than 5% of the AI calls failed.

In [13]:
EXPECTED_SCHEMA = [
    ("drawing_id", "string"), ("title_block_number", "string"), ("discipline", "string"), ("revision", "string"), ("drawing_type", "string"),
    ("spec_citations", "array<string>"), ("extract_status", "string"), ("classify_status", "string"), ("extracted_at", "timestamp"),
]
assert spark.table(TARGET).dtypes == EXPECTED_SCHEMA, spark.table(TARGET).dtypes

disciplines_sql = ", ".join(f"'{d}'" for d in DISCIPLINES)
labels_sql = ", ".join(f"'{label}'" for label in LABELS)
counts_df = spark.sql(f"""
    SELECT COUNT(*) AS drawings,
           COUNT(DISTINCT drawing_id) AS distinct_ids,
           COUNT_IF(extract_status = 'ok') AS extract_ok,
           COUNT_IF(classify_status = 'ok') AS classify_ok,
           COUNT_IF(discipline IS NOT NULL) AS with_discipline,
           COUNT_IF(discipline IS NOT NULL AND discipline NOT IN ({disciplines_sql})) AS bad_discipline,
           COUNT_IF(classify_status = 'ok' AND (drawing_type IS NULL OR drawing_type NOT IN ({labels_sql}))) AS bad_type,
           COUNT_IF(spec_citations IS NULL) AS null_citations,
           COUNT_IF(extracted_at IS NULL) AS no_timestamp
    FROM {TARGET}
""")
display(counts_df)
display(spark.sql(f"SELECT extract_status, classify_status, COUNT(*) AS drawings FROM {TARGET} GROUP BY extract_status, classify_status ORDER BY drawings DESC"))

c = counts_df.first()
orphans = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {TARGET} t WHERE NOT EXISTS (SELECT 1 FROM {PARSED} p WHERE p.drawing_id = t.drawing_id)
""").first().n
not_extracted = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {PARSED} p WHERE p.parse_status = 'ok' AND NOT EXISTS (SELECT 1 FROM {TARGET} t WHERE t.drawing_id = p.drawing_id)
""").first().n
print(f"{c.drawings} rows, {c.distinct_ids} distinct drawing_id, {orphans} without a drawings_parsed row, "
      f"{not_extracted} parsed drawings without a row ({'expected: a trial run' if MAX_ROWS else 'none expected'})")
print(f"extract_status ok: {c.extract_ok} ({c.extract_ok / c.drawings:.1%}); classify_status ok: {c.classify_ok} ({c.classify_ok / c.drawings:.1%}); "
      f"discipline set: {c.with_discipline} ({c.with_discipline / c.drawings:.1%})")

assert c.drawings == n_docs and c.distinct_ids == n_docs, "one row per drawing sent"
if not MAX_ROWS:
    assert c.drawings == 1200, "the corpus has 1,200 drawings"
assert orphans == 0, "a row has no drawing in drawings_parsed"
assert not_extracted == n_ok - n_docs, "a parsed drawing is missing from the table"
assert c.bad_discipline == 0 and c.bad_type == 0, "a value outside the allowed sets"
assert c.null_citations == 0 and c.no_timestamp == 0
assert c.extract_ok >= 0.95 * c.drawings and c.classify_ok >= 0.95 * c.drawings, "more than 5% of the AI calls failed; read extract_status and classify_status"
assert c.with_discipline >= 0.95 * c.drawings, "discipline is missing for more than 5% of the drawings"
print("all checks passed")

drawings,distinct_ids,extract_ok,classify_ok,with_discipline,bad_discipline,bad_type,null_citations,no_timestamp
1200,1200,1200,1200,1198,0,0,0,0


extract_status,classify_status,drawings
ok,ok,1200


1200 rows, 1200 distinct drawing_id, 0 without a drawings_parsed row, 0 parsed drawings without a row (none expected)
extract_status ok: 1200 (100.0%); classify_status ok: 1200 (100.0%); discipline set: 1198 (99.8%)
all checks passed


## What came out

The mix of drawing types and disciplines, the two together, and the citations. A drawing with a NULL `drawing_type` or `discipline` is one whose call failed or whose word was not recognised.

In [14]:
display(spark.sql(f"""
    SELECT drawing_type, COUNT(*) AS drawings, ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM {TARGET} GROUP BY drawing_type ORDER BY drawings DESC
"""))
display(spark.sql(f"""
    SELECT discipline, COUNT(*) AS drawings, ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM {TARGET} GROUP BY discipline ORDER BY drawings DESC
"""))
print("discipline (rows) by drawing_type (columns):")
display(spark.table(TARGET).groupBy("discipline").pivot("drawing_type", LABELS).count().fillna(0).orderBy("discipline"))

cites = spark.sql(f"""
    SELECT COUNT(*) AS drawings,
           ROUND(AVG(size(spec_citations)), 2) AS avg_citations,
           COUNT_IF(size(spec_citations) >= 1) AS with_citations,
           ROUND(100 * COUNT_IF(size(spec_citations) >= 1) / COUNT(*), 1) AS pct_with_citations,
           MAX(size(spec_citations)) AS max_citations
    FROM {TARGET}
""")
display(cites)
print("the most cited:")
display(spark.sql(f"""
    SELECT citation, COUNT(*) AS drawings
    FROM (SELECT explode(spec_citations) AS citation FROM {TARGET})
    GROUP BY citation ORDER BY drawings DESC, citation LIMIT 12
"""))

drawing_type,drawings,pct
ISO,397,33.1
PID,373,31.1
GA,269,22.4
SPEC,135,11.3
OTHER,26,2.2


discipline,drawings,pct
piping,755,62.9
mechanical,134,11.2
instr,119,9.9
structural,113,9.4
electrical,77,6.4
,2,0.2


discipline (rows) by drawing_type (columns):


discipline,ISO,PID,GA,SPEC,OTHER
,1,1,0,0,0
electrical,0,0,71,0,6
instr,0,118,0,0,1
mechanical,0,0,90,40,4
piping,396,254,0,95,10
structural,0,0,108,0,5


drawings,avg_citations,with_citations,pct_with_citations,max_citations
1200,3.87,1200,100.0,6


the most cited:


citation,drawings
ASME B16.5,904
OSHA 29 CFR 1910.147,641
ASME B31.3,537
ASME B31.3 §341,516
ASME B31.3 §322,448
ISA 5.1,398
ASME B16.9,396
NFPA 59A,267
IBC 2021 CHAPTER 10,250
IBC 1005.1,140


## Against the generator's records

The synthetic data generator wrote down each drawing's real discipline, revision and type in `drawing_manifest.json`, and the planted flaws' code citations in `ground_truth_findings.jsonl`. These are only for judging the models' answers; nothing above used them. The title block number should equal the drawing id, and a planted flaw's standard should be among the standards its drawing cites.

In [15]:
MANIFEST = Path("../01_synthetic_data/outputs/drawing_manifest.json")  # same relative path in the repo and in a bundle deploy
GROUND_TRUTH = Path("../01_synthetic_data/outputs/ground_truth_findings.jsonl")
STANDARD = r"^(ASME B[0-9.]+|IBC|OSHA|NFPA [0-9A-Z]+|ISA [0-9.]+|API [0-9]+)"  # the standard a citation belongs to, without its clause

if not MANIFEST.exists():
    print(f"{MANIFEST} not found: skipping the comparison")
else:
    truth = spark.createDataFrame(
        [(d["drawing_id"], d["discipline"], d["revision"], d["drawing_type"], d["has_flaw"]) for d in json.loads(MANIFEST.read_text())],
        "drawing_id string, true_discipline string, true_revision string, true_type string, has_flaw boolean",
    )
    truth.createOrReplaceTempView("truth")
    spark.table(TARGET).join(truth, "drawing_id").createOrReplaceTempView("compared")
    display(spark.sql("""
        SELECT COUNT(*) AS drawings,
               COUNT_IF(title_block_number = drawing_id) AS number_is_id,
               COUNT_IF(discipline = true_discipline) AS discipline_right,
               COUNT_IF(revision = true_revision) AS revision_right,
               COUNT_IF(drawing_type = true_type) AS type_right
        FROM compared
    """))
    print("drawing_type by the type the generator made:")
    display(spark.sql("""
        SELECT true_type, COUNT(*) AS drawings, COUNT_IF(drawing_type = true_type) AS correct, COUNT_IF(drawing_type IS NULL OR drawing_type <> true_type) AS wrong
        FROM compared GROUP BY true_type ORDER BY drawings DESC
    """))
    print("drawings where any of the four differs (up to 15):")
    display(spark.sql("""
        SELECT drawing_id, title_block_number, true_type, drawing_type, true_discipline, discipline, true_revision, revision
        FROM compared
        WHERE NOT (title_block_number <=> drawing_id AND drawing_type <=> true_type AND discipline <=> true_discipline AND revision <=> true_revision)
        ORDER BY drawing_id LIMIT 15
    """))

    if GROUND_TRUTH.exists():
        planted = spark.createDataFrame(
            [(f["drawing_id"], f["citation"]) for f in map(json.loads, GROUND_TRUTH.read_text().splitlines())], "drawing_id string, citation string"
        )
        planted.join(spark.table(TARGET).select("drawing_id", "spec_citations"), "drawing_id").createOrReplaceTempView("planted")
        print("the planted flaws on these drawings, against the standards each drawing cites:")
        display(spark.sql(f"""
            SELECT COUNT(*) AS planted_flaws,
                   COUNT(DISTINCT drawing_id) AS flawed_drawings,
                   COUNT_IF(array_contains(spec_citations, citation)) AS citation_listed_exactly,
                   COUNT_IF(regexp_extract(citation, '{STANDARD}', 1) <> '' AND
                            exists(spec_citations, s -> regexp_extract(s, '{STANDARD}', 1) = regexp_extract(citation, '{STANDARD}', 1))) AS standard_listed
            FROM planted
        """))
        flawed = spark.sql(f"""
            SELECT COUNT(*) AS flawed_drawings, COUNT_IF(size(t.spec_citations) >= 1) AS with_citations
            FROM {TARGET} t JOIN truth m ON t.drawing_id = m.drawing_id WHERE m.has_flaw
        """).first()
        print(f"{flawed.with_citations} of {flawed.flawed_drawings} flawed drawings have at least one citation")

drawings,number_is_id,discipline_right,revision_right,type_right
1200,1193,1198,1200,1200


drawing_type by the type the generator made:


true_type,drawings,correct,wrong
ISO,397,397,0
PID,373,373,0
GA,269,269,0
SPEC,135,135,0
OTHER,26,26,0


drawings where any of the four differs (up to 15):


drawing_id,title_block_number,true_type,drawing_type,true_discipline,discipline,true_revision,revision
BRL-INS-PID-0972,BRL-INS-PID-0972,PID,PID,instr,,C,C
BRL-PIP-ISO-0119,BRL-PIP-ISO-0010,ISO,ISO,piping,piping,C,C
BRL-PIP-ISO-0214,BRL-PIP-ISO-00214,ISO,ISO,piping,piping,D,D
BRL-PIP-ISO-0238,BRL-PIP-ISO-00238,ISO,ISO,piping,piping,D,D
BRL-PIP-ISO-0526,BRL-PIP-ISO-00526,ISO,ISO,piping,piping,D,D
BRL-PIP-ISO-0664,BRL-PIP-ISO-00664,ISO,ISO,piping,piping,D,D
BRL-PIP-ISO-0687,BRL-PIP-ISO-00687,ISO,ISO,piping,piping,E,E
BRL-PIP-ISO-0698,BRL-PIP-ISO-00698,ISO,ISO,piping,piping,C,C
BRL-PIP-ISO-1146,BRL-PIP-ISO-1146,ISO,ISO,piping,,D,D


the planted flaws on these drawings, against the standards each drawing cites:


planted_flaws,flawed_drawings,citation_listed_exactly,standard_listed
274,240,122,274


240 of 240 flawed drawings have at least one citation


## A sample for the repository

Ten rows written to `outputs/sample_extracted.jsonl`, one JSON object per line. The mix is picked by hand to cover all five disciplines and all five drawing types; for each (discipline, type) pair the first drawing id in the table that was extracted and classified without error and cites at least two standards is used, and the P&ID that the parse notebook wrote out in full stands for `instr` / `PID`.

In [16]:
SAMPLE_MIX = [  # (discipline, drawing_type)
    ("piping", "ISO"), ("piping", "PID"), ("instr", "PID"), ("structural", "GA"), ("mechanical", "GA"),
    ("electrical", "GA"), ("piping", "SPEC"), ("mechanical", "SPEC"), ("electrical", "OTHER"), ("structural", "OTHER"),
]
usable = spark.table(TARGET).where("extract_status = 'ok' AND classify_status = 'ok' AND size(spec_citations) >= 2").orderBy("drawing_id").collect()
picked, missing = [], []
for discipline, drawing_type in SAMPLE_MIX:
    matches = [r for r in usable if r.discipline == discipline and r.drawing_type == drawing_type]
    chosen = next((r for r in matches if r.drawing_id == SAMPLE_ID), matches[0] if matches else None)
    if chosen is None:
        missing.append((discipline, drawing_type))
    else:
        picked.append(chosen)
if missing:  # only in a trial run, or if the models left a pair empty: fill up with the earliest other drawings
    print("no drawing for", missing, "- filling up with other drawings")
    picked += [r for r in usable if r.drawing_id not in {p.drawing_id for p in picked}][: len(missing)]
if not MAX_ROWS:
    assert len(picked) == 10 and len({r.drawing_id for r in picked}) == 10, "expected ten distinct sample rows"

utc = spark.conf.get("spark.sql.session.timeZone") in ("UTC", "Etc/UTC")  # serverless sessions run in UTC
lines = []
for r in picked:
    lines.append(json.dumps({
        "drawing_id": r.drawing_id,
        "title_block_number": r.title_block_number,
        "discipline": r.discipline,
        "revision": r.revision,
        "drawing_type": r.drawing_type,
        "spec_citations": list(r.spec_citations),
        "extract_status": r.extract_status,
        "classify_status": r.classify_status,
        "extracted_at": r.extracted_at.strftime("%Y-%m-%dT%H:%M:%S") + ("Z" if utc else ""),
    }, ensure_ascii=False))

OUT.mkdir(exist_ok=True)
(OUT / "sample_extracted.jsonl").write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"wrote {OUT / 'sample_extracted.jsonl'}: {len(lines)} rows\n")
print("\n".join(lines))

wrote outputs/sample_extracted.jsonl: 10 rows

{"drawing_id": "BRL-PIP-ISO-0004", "title_block_number": "BRL-PIP-ISO-0004", "discipline": "piping", "revision": "B", "drawing_type": "ISO", "spec_citations": ["ASME B31.3 §341", "ASME B16.5", "ASME B16.9"], "extract_status": "ok", "classify_status": "ok", "extracted_at": "2026-11-07T05:47:11Z"}
{"drawing_id": "BRL-PIP-PID-0009", "title_block_number": "BRL-PIP-PID-0009", "discipline": "piping", "revision": "E", "drawing_type": "PID", "spec_citations": ["ASME B31.3 §322", "ASME B16.5", "OSHA 29 CFR 1910.147", "ISA 5.1"], "extract_status": "ok", "classify_status": "ok", "extracted_at": "2026-11-07T05:47:11Z"}
{"drawing_id": "BRL-INS-PID-0052", "title_block_number": "BRL-INS-PID-0052", "discipline": "instr", "revision": "B", "drawing_type": "PID", "spec_citations": ["ASME B31.3 §322", "ASME B16.5", "OSHA 29 CFR 1910.147", "ISA 5.1"], "extract_status": "ok", "classify_status": "ok", "extracted_at": "2026-11-07T05:47:11Z"}
{"drawing_id": "BRL-S

`drawings_extracted` is the input of the later notebooks that need a drawing's discipline, revision, type or citations: `03_compliance_check` will join to it on `drawing_id` and carry those columns into `compliance_findings`. `drawings_extracted_raw` keeps the models' answers, so the cleaning rules can be changed and the table rebuilt without sending a single drawing to a model again.